In [18]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.varmax import VARMAX
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
from tqdm import tqdm_notebook
from itertools import product
import math

import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np

df = pd.read_csv('tokyo_weather.csv')
df.head()

if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.sort_values("Date")
    x = df["Date"]
else:
    x = df.index

for col in df.columns:
    if col != "Date":
        df[col] = pd.to_numeric(df[col], errors="coerce")

numeric_cols = df.select_dtypes(include="number").columns

n_cols = 2
n_rows = math.ceil(len(numeric_cols) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(14, 3 * n_rows)
)

axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].plot(x, df[col], color="red")
    axes[i].set_title(col, fontsize=14)
    axes[i].tick_params(axis="x", rotation=45)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'tqdm'

In [ ]:
df.head()

,Date,Average Temperature (°C),Highest Temperature (°C),Highest Temperature (°C) Datetime,Lowest Temperature (°C),Lowest Temperature (°C) Datetime,Total Precipitation (mm),Sunshine Duration (hours),Maximum Snow Depth (cm),Maximum Snow Depth (cm) Datetime,...,Maximum Wind Speed (m/s) Datetime,Maximum Wind Speed (m/s) Direction,Maximum Gust Speed (m/s),Maximum Gust Speed (m/s) Datetime,Maximum Gust Speed (m/s) Direction,Most Frequent Wind Direction (16-point compass),Average Vapor Pressure (hPa),Average Humidity (%),Minimum Relative Humidity (%),Minimum Relative Humidity (%) Datetime
0,2018-06-26,25.7,30.1,NaN,22.3,NaN,0.0,9.2,0,NaN,...,NaN,NaN,12.4,NaN,NaN,NaN,24.7,75,56,NaN
1,2018-06-27,27.7,31.7,NaN,24.9,NaN,0.0,7.6,0,NaN,...,NaN,NaN,20.0,NaN,NaN,NaN,26.1,71,57,NaN
2,2018-06-28,27.4,31.9,NaN,25.2,NaN,0.0,4.8,0,NaN,...,NaN,NaN,16.1,NaN,NaN,NaN,27.9,77,60,NaN
3,2018-06-29,28.2,32.9,NaN,25.4,NaN,0.0,12.8,0,NaN,...,NaN,NaN,16.9,NaN,NaN,NaN,27.7,73,55,NaN
4,2018-06-30,28.6,32.7,NaN,25.2,NaN,0.0,13.1,0,NaN,...,NaN,NaN,15.5,NaN,NaN,NaN,28.5,74,57,NaN


In [ ]:
import pandas as pd
import numpy as np
import warnings
import contextlib
import io

from statsmodels.tsa.stattools import grangercausalitytests

# -------------------------------------------------------
# Load data
# -------------------------------------------------------

df = pd.read_csv("tokyo_weather.csv")

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.sort_values("Date").reset_index(drop=True)

# -------------------------------------------------------
# Keep only numeric weather variables
# -------------------------------------------------------

data = df.select_dtypes(include=[np.number]).copy()

data = data.dropna(axis=1, how="all")
data = data.loc[:, data.nunique(dropna=True) > 1]

data = data.interpolate(limit_direction="both")
data = data.ffill().bfill()

print("Variables used:")
print(data.columns.tolist())

# -------------------------------------------------------
# Set target variable
# -------------------------------------------------------

target_variable = "Highest Temperature (°C)"

if target_variable not in data.columns:
    raise ValueError(f"{target_variable} not found in data columns.")

# -------------------------------------------------------
# Optional: difference the data
# -------------------------------------------------------

USE_DIFFERENCE = True

if USE_DIFFERENCE:
    data_for_test = data.diff().dropna()
else:
    data_for_test = data.copy()

# -------------------------------------------------------
# Run Granger tests ONLY for hottest temperature
# -------------------------------------------------------

max_lag = 7

results_list = []
errors = []

# Only test predictors for Highest Temperature
predictors = [col for col in data_for_test.columns if col != target_variable]

for predictor in predictors:

    pair_data = data_for_test[[target_variable, predictor]].dropna()

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            with contextlib.redirect_stdout(io.StringIO()):
                result = grangercausalitytests(
                    pair_data,
                    maxlag=max_lag,
                    verbose=False
                )

        for lag in range(1, max_lag + 1):

            f_test = result[lag][0]["ssr_ftest"]

            f_stat = f_test[0]
            p_value = f_test[1]

            results_list.append({
                "Target / caused variable (Y)": target_variable,
                "Predictor / causing variable (X)": predictor,
                "Lag": lag,
                "F statistic": f_stat,
                "p-value": p_value,
                "Significant at 5%": p_value < 0.05,
                "Interpretation": (
                    f"{predictor} may help predict {target_variable}"
                    if p_value < 0.05
                    else f"{predictor} is not useful for predicting {target_variable}"
                )
            })

    except Exception as e:
        errors.append({
            "Target": target_variable,
            "Predictor": predictor,
            "Error": str(e)
        })

# -------------------------------------------------------
# Full results for hottest temperature only
# -------------------------------------------------------

granger_results = pd.DataFrame(results_list)

granger_results = granger_results.sort_values("p-value").reset_index(drop=True)

display(granger_results)

# -------------------------------------------------------
# Keep only useful/significant predictors
# -------------------------------------------------------

useful_results = granger_results[granger_results["p-value"] < 0.05].copy()

useful_results = useful_results.sort_values("p-value").reset_index(drop=True)

display(useful_results)

# -------------------------------------------------------
# Best lag for each useful predictor
# -------------------------------------------------------

best_useful_predictors = (
    useful_results
    .sort_values("p-value")
    .groupby("Predictor / causing variable (X)", as_index=False)
    .first()
)

best_useful_predictors = best_useful_predictors.rename(columns={
    "Lag": "Best lag",
    "F statistic": "Best F statistic",
    "p-value": "Best p-value"
})

best_useful_predictors = best_useful_predictors.sort_values("Best p-value").reset_index(drop=True)

display(best_useful_predictors)

# -------------------------------------------------------
# Save results
# -------------------------------------------------------

granger_results.to_csv("granger_hottest_temp_all_lags.csv", index=False)
useful_results.to_csv("granger_hottest_temp_useful_all_lags.csv", index=False)
best_useful_predictors.to_csv("granger_hottest_temp_best_useful_predictors.csv", index=False)

# -------------------------------------------------------
# Print summary
# -------------------------------------------------------

print("Target variable:", target_variable)
print("Number of predictors tested:", len(predictors))
print("Number of total test rows:", len(granger_results))
print("Number of useful/significant rows:", len(useful_results))
print("Number of useful predictors:", len(best_useful_predictors))
print("Number of errors:", len(errors))

if errors:
    errors_df = pd.DataFrame(errors)
    display(errors_df)

Variables used:
['Average Temperature (°C)', 'Highest Temperature (°C)', 'Lowest Temperature (°C)', 'Total Precipitation (mm)', 'Sunshine Duration (hours)', 'Maximum Snow Depth (cm)', 'Total Snowfall (cm)', 'Average Wind Speed (m/s)', 'Maximum Wind Speed (m/s)', 'Maximum Gust Speed (m/s)', 'Average Vapor Pressure (hPa)', 'Average Humidity (%)', 'Minimum Relative Humidity (%)']


,Target / caused variable (Y),Predictor / causing variable (X),Lag,F statistic,p-value,Significant at 5%,Interpretation
0,Highest Temperature (°C),Average Vapor Pressure (hPa),3,47.356987,1.202588e-29,True,Average Vapor Pressure (hPa) may help predict ...
1,Highest Temperature (°C),Average Vapor Pressure (hPa),2,67.537564,3.465068e-29,True,Average Vapor Pressure (hPa) may help predict ...
2,Highest Temperature (°C),Average Vapor Pressure (hPa),6,25.188722,4.818141e-29,True,Average Vapor Pressure (hPa) may help predict ...
3,Highest Temperature (°C),Average Vapor Pressure (hPa),5,29.243111,8.137382e-29,True,Average Vapor Pressure (hPa) may help predict ...
4,Highest Temperature (°C),Average Vapor Pressure (hPa),4,35.178926,1.647160e-28,True,Average Vapor Pressure (hPa) may help predict ...
...,...,...,...,...,...,...,...
79,Highest Temperature (°C),Total Snowfall (cm),5,0.318887,9.018597e-01,False,Total Snowfall (cm) is not useful for predicti...
80,Highest Temperature (°C),Total Snowfall (cm),4,0.253455,9.076581e-01,False,Total Snowfall (cm) is not useful for predicti...
81,Highest Temperature (°C),Maximum Snow Depth (cm),6,0.327293,9.229185e-01,False,Maximum Snow Depth (cm) is not useful for pred...
82,Highest Temperature (°C),Total Snowfall (cm),7,0.346188,9.326559e-01,False,Total Snowfall (cm) is not useful for predicti...


,Target / caused variable (Y),Predictor / causing variable (X),Lag,F statistic,p-value,Significant at 5%,Interpretation
0,Highest Temperature (°C),Average Vapor Pressure (hPa),3,47.356987,1.202588e-29,True,Average Vapor Pressure (hPa) may help predict ...
1,Highest Temperature (°C),Average Vapor Pressure (hPa),2,67.537564,3.465068e-29,True,Average Vapor Pressure (hPa) may help predict ...
2,Highest Temperature (°C),Average Vapor Pressure (hPa),6,25.188722,4.818141e-29,True,Average Vapor Pressure (hPa) may help predict ...
3,Highest Temperature (°C),Average Vapor Pressure (hPa),5,29.243111,8.137382e-29,True,Average Vapor Pressure (hPa) may help predict ...
4,Highest Temperature (°C),Average Vapor Pressure (hPa),4,35.178926,1.647160e-28,True,Average Vapor Pressure (hPa) may help predict ...
...,...,...,...,...,...,...,...
63,Highest Temperature (°C),Total Precipitation (mm),5,2.849895,1.431268e-02,True,Total Precipitation (mm) may help predict High...
64,Highest Temperature (°C),Sunshine Duration (hours),2,4.137061,1.609498e-02,True,Sunshine Duration (hours) may help predict Hig...
65,Highest Temperature (°C),Sunshine Duration (hours),5,2.502193,2.872829e-02,True,Sunshine Duration (hours) may help predict Hig...
66,Highest Temperature (°C),Sunshine Duration (hours),6,2.282642,3.360522e-02,True,Sunshine Duration (hours) may help predict Hig...


,Predictor / causing variable (X),Target / caused variable (Y),Best lag,Best F statistic,Best p-value,Significant at 5%,Interpretation
0,Average Vapor Pressure (hPa),Highest Temperature (°C),3,47.356987,1.202588e-29,True,Average Vapor Pressure (hPa) may help predict ...
1,Average Temperature (°C),Highest Temperature (°C),6,22.284182,1.367799e-25,True,Average Temperature (°C) may help predict High...
2,Minimum Relative Humidity (%),Highest Temperature (°C),3,37.922946,6.930084e-24,True,Minimum Relative Humidity (%) may help predict...
3,Average Humidity (%),Highest Temperature (°C),1,97.656680,1.465281e-22,True,Average Humidity (%) may help predict Highest ...
4,Lowest Temperature (°C),Highest Temperature (°C),3,26.475742,8.105263e-17,True,Lowest Temperature (°C) may help predict Highe...
5,Average Wind Speed (m/s),Highest Temperature (°C),2,10.093733,4.329863e-05,True,Average Wind Speed (m/s) may help predict High...
6,Maximum Gust Speed (m/s),Highest Temperature (°C),7,4.206518,1.285568e-04,True,Maximum Gust Speed (m/s) may help predict High...
7,Sunshine Duration (hours),Highest Temperature (°C),4,5.243212,3.341721e-04,True,Sunshine Duration (hours) may help predict Hig...
8,Total Precipitation (mm),Highest Temperature (°C),6,3.775725,9.589536e-04,True,Total Precipitation (mm) may help predict High...
9,Maximum Wind Speed (m/s),Highest Temperature (°C),3,5.175248,1.453070e-03,True,Maximum Wind Speed (m/s) may help predict High...


Target variable: Highest Temperature (°C)
Number of predictors tested: 12
Number of total test rows: 84
Number of useful/significant rows: 68
Number of useful predictors: 10
Number of errors: 0


In [ ]:

from statsmodels.tsa.api import VAR

train_df = df[:-15]
test_df = df[-15:]

pred_list = best_useful_predictors['Predictor / causing variable (X)'].tolist()

train_X = train_df[pred_list]
test_X = test_df[pred_list]



var_model = VAR(train_X.diff()[1:])

fitted_model = var_model.fit(disp=False)

NameError: name 'var_model' is not defined